# Phase 1 - Data selection, audit, and initial feature engineering
ACN-Data (Caltech + JPL) -> 15-min occupancy series ("cars plugged in at time t")

In [2]:
"""
Phase 1 - Data selection, audit, and initial feature engineering
ACN-Data (Caltech + JPL) -> 15-min occupancy series ("cars plugged in at time t")

Usage:
    python phase1_data_pipeline.py --data_dir ./data --out_dir ./outputs
Outputs (in out_dir): tables/*.csv, figures/*.png, occupancy_features.parquet
"""
import argparse, glob, json, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FREQ = "15min"
MAX_KWH = 100                # no passenger EV battery delivers more in one session


# ------------------------------------------------------------------ loading
def load_file(path):
    with open(path) as f:
        d = json.load(f)
    items = d["_items"] if isinstance(d, dict) and "_items" in d else d
    return pd.DataFrame(items)


def load_all(data_dir):
    frames = []
    for p in sorted(glob.glob(os.path.join(data_dir, "acn_*.json"))):
        df = load_file(p)
        df["site"] = "caltech" if "caltech" in os.path.basename(p) else "jpl"
        df["source_file"] = os.path.basename(p)
        frames.append(df)
    df = pd.concat(frames, ignore_index=True)
    return df


def flatten_user_inputs(df):
    """userInputs is a list of dicts (user may edit it several times). Keep the LAST entry
    and the number of edits; sessions with no userInputs get NaN."""
    def last(x):
        return x[-1] if isinstance(x, list) and len(x) else {}
    ui = df["userInputs"].apply(last).apply(pd.Series)
    ui = ui.add_prefix("ui_")
    df = df.drop(columns=["userInputs"]).join(ui)
    df["n_userinput_edits"] = df.get("userInputs", pd.Series(dtype=object))
    return df


# ------------------------------------------------------------------ cleaning
def parse_times(df):
    for c in ["connectionTime", "disconnectTime", "doneChargingTime", "ui_requestedDeparture", "ui_modifiedAt"]:
        if c in df:
            df[c] = pd.to_datetime(df[c], utc=True, errors="coerce")
    return df


def add_flags(df):
    df["duration_h"] = (df["disconnectTime"] - df["connectionTime"]).dt.total_seconds() / 3600
    df["charge_h"] = (df["doneChargingTime"] - df["connectionTime"]).dt.total_seconds() / 3600
    df["has_userinputs"] = df["ui_kWhRequested"].notna() if "ui_kWhRequested" in df else False
    df["flag_neg_duration"] = df["duration_h"] <= 0
    df["flag_kwh_outlier"] = (df["kWhDelivered"] > MAX_KWH) | (df["kWhDelivered"] < 0)
    df["flag_done_after_disc"] = df["doneChargingTime"] > df["disconnectTime"]
    return df


def clean(df):
    key = [c for c in ["sessionID"] if c in df]
    df = df.drop_duplicates(subset=key, keep="first")
    bad = df[["flag_neg_duration", "flag_kwh_outlier"]].any(axis=1)
    return df[~bad].copy()


# ------------------------------------------------------------------ audit tables
def audit(df_raw, df_clean, out):
    os.makedirs(f"{out}/tables", exist_ok=True)

    miss = (df_raw.drop(columns=["_id"], errors="ignore")
            .isna().mean().mul(100).round(2).sort_values(ascending=False).rename("pct_missing"))
    miss.to_csv(f"{out}/tables/missingness.csv")

    num_cols = ["kWhDelivered", "duration_h", "charge_h",
                "ui_milesRequested", "ui_WhPerMile", "ui_minutesAvailable", "ui_kWhRequested"]
    num_cols = [c for c in num_cols if c in df_raw]
    df_raw[num_cols].describe(percentiles=[.01, .25, .5, .75, .99]).T.round(3).to_csv(f"{out}/tables/summary_stats.csv")

    flags = [c for c in df_raw if c.startswith("flag_")]
    report = {
        "rows_raw": len(df_raw),
        "duplicate_sessionIDs": int(df_raw.duplicated("sessionID").sum()),
        **{f: int(df_raw[f].sum()) for f in flags},
        "rows_clean": len(df_clean),
        "pct_with_userinputs": round(100 * df_raw["has_userinputs"].mean(), 2),
    }
    pd.Series(report).to_csv(f"{out}/tables/quality_report.csv", header=["value"])
    df_clean.head(5).T.to_csv(f"{out}/tables/example_rows.csv")

    per_site_year = (df_clean.assign(year=df_clean["connectionTime"].dt.year)
                     .groupby(["site", "year"]).size().rename("sessions"))
    per_site_year.to_csv(f"{out}/tables/sessions_per_site_year.csv")
    print(pd.Series(report))


# ------------------------------------------------------------------ plots
def plots(df, occ, out):
    os.makedirs(f"{out}/figures", exist_ok=True)

    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    df["duration_h"].clip(upper=24).hist(bins=60, ax=ax[0]); ax[0].set_title("Session duration (h, clipped 24)")
    df["kWhDelivered"].hist(bins=60, ax=ax[1]); ax[1].set_title("kWh delivered")
    df["site"].value_counts().plot.bar(ax=ax[2]); ax[2].set_title("Sessions per site")
    plt.tight_layout(); plt.savefig(f"{out}/figures/distributions.png", dpi=150); plt.close()

    # daily profile + weekly profile of occupancy
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    for site, s in occ.items():
        loc = s.tz_convert("America/Los_Angeles")
        wd = loc[loc.index.dayofweek < 5]
        wd.groupby(wd.index.hour + wd.index.minute / 60).mean().plot(ax=ax[0], label=site)
        loc.groupby(loc.index.dayofweek).mean().plot.bar(ax=ax[1], alpha=.6) if site == "caltech" else None
    ax[0].set_title("Mean weekday occupancy by hour (local)"); ax[0].legend()
    ax[1].set_title("Caltech mean occupancy by day of week (0=Mon)")
    plt.tight_layout(); plt.savefig(f"{out}/figures/occupancy_profiles.png", dpi=150); plt.close()

    # drift: monthly mean occupancy
    fig, ax = plt.subplots(figsize=(12, 4))
    for site, s in occ.items():
        s.resample("MS").mean().plot(ax=ax, label=site)
    ax.set_title("Monthly mean occupancy (drift / COVID / capacity changes)"); ax.legend()
    plt.tight_layout(); plt.savefig(f"{out}/figures/occupancy_drift.png", dpi=150); plt.close()

    # correlation heatmap
    cols = [c for c in ["kWhDelivered", "duration_h", "charge_h", "ui_kWhRequested",
                        "ui_minutesAvailable", "ui_milesRequested", "ui_WhPerMile"] if c in df]
    corr = df[cols].corr()
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=45, ha="right")
    ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols)
    for i in range(len(cols)):
        for j in range(len(cols)):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
    plt.colorbar(im); plt.tight_layout()
    plt.savefig(f"{out}/figures/correlation.png", dpi=150); plt.close()


# ------------------------------------------------------------------ target + features
def occupancy_series(d, freq=FREQ):
    """Number of cars plugged in at each grid timestamp, for one site."""
    start = d["connectionTime"].min().floor(freq)
    end = d["disconnectTime"].max().ceil(freq)
    grid = pd.date_range(start, end, freq=freq)
    arr = d["connectionTime"].dt.floor(freq).value_counts()
    dep = d["disconnectTime"].dt.ceil(freq).value_counts()
    delta = arr.reindex(grid, fill_value=0) - dep.reindex(grid, fill_value=0)
    return delta.cumsum().rename("occupancy")


def build_features(occ, df_site, horizon_steps=4):
    """horizon_steps=4 -> predict 1 h ahead. All features use information up to t only."""
    X = pd.DataFrame({"y": occ.shift(-horizon_steps)})   # target: occupancy at t+h
    X["occ_t"] = occ
    loc = occ.index.tz_convert("America/Los_Angeles")
    frac_hour = loc.hour + loc.minute / 60
    X["hour_sin"] = np.sin(2 * np.pi * frac_hour / 24)
    X["hour_cos"] = np.cos(2 * np.pi * frac_hour / 24)
    X["dow_sin"] = np.sin(2 * np.pi * loc.dayofweek / 7)
    X["dow_cos"] = np.cos(2 * np.pi * loc.dayofweek / 7)
    X["month_sin"] = np.sin(2 * np.pi * loc.month / 12)
    X["month_cos"] = np.cos(2 * np.pi * loc.month / 12)
    X["is_weekend"] = (loc.dayofweek >= 5).astype(int)
    try:
        import holidays
        hol = holidays.US(state="CA")
        X["is_holiday"] = [int(d in hol) for d in loc.date]
    except ImportError:
        X["is_holiday"] = 0

    steps_day, steps_week = 96, 672
    for lag in [1, 2, 4, steps_day, steps_week]:
        X[f"lag_{lag}"] = occ.shift(lag)
    X["roll_mean_4"] = occ.rolling(4).mean()
    X["roll_mean_96"] = occ.rolling(steps_day).mean()
    X["roll_std_4"] = occ.rolling(4).std()
    X["diff_1"] = occ.diff()

    # flows: arrivals / departures in last hour (computed from past events only)
    grid = occ.index
    arr = df_site["connectionTime"].dt.floor(FREQ).value_counts().reindex(grid, fill_value=0)
    dep = df_site["disconnectTime"].dt.floor(FREQ).value_counts().reindex(grid, fill_value=0)
    X["arrivals_1h"] = arr.rolling(4).sum()
    X["departures_1h"] = dep.rolling(4).sum()

    # plugged-in cars that are already done charging at t (known at t)
    done = df_site["doneChargingTime"].dt.floor(FREQ).value_counts().reindex(grid, fill_value=0).cumsum()
    done_dep = df_site.loc[df_site["doneChargingTime"].notna(), "disconnectTime"].dt.floor(FREQ) \
        .value_counts().reindex(grid, fill_value=0).cumsum()
    X["n_done_charging_plugged"] = (done - done_dep).clip(lower=0)
    return X


# ------------------------------------------------------------------ main
data_dir = "acn_dataset"
out_dir = "outputs"

os.makedirs(out_dir, exist_ok=True)
df = load_all(data_dir)
df = flatten_user_inputs(df)
df = parse_times(df)
df = add_flags(df)
df_clean = clean(df)
audit(df, df_clean, out_dir)

occ = {s: occupancy_series(g) for s, g in df_clean.groupby("site")}
plots(df_clean, occ, out_dir)

feats = []
for s, g in df_clean.groupby("site"):
    f = build_features(occ[s], g)
    f["site"] = s
    feats.append(f)
feats = pd.concat(feats).dropna(subset=["y", "lag_672"])
feats.to_parquet(f"{out_dir}/occupancy_features.parquet")
print("feature table:", feats.shape)

rows_raw                44989.00
duplicate_sessionIDs        0.00
flag_neg_duration           0.00
flag_kwh_outlier            1.00
flag_done_after_disc     3549.00
rows_clean              44988.00
pct_with_userinputs        93.06
dtype: float64
feature table: (188106, 23)


In [3]:
import pandas as pd

df = pd.read_parquet("outputs/occupancy_features.parquet")

print(df.head())
print(df.info())
print(df.shape) 

                             y  occ_t  hour_sin  hour_cos  dow_sin  dow_cos  \
2019-01-08 05:15:00+00:00  1.0      2 -0.659346  0.751840      0.0      1.0   
2019-01-08 05:30:00+00:00  1.0      1 -0.608761  0.793353      0.0      1.0   
2019-01-08 05:45:00+00:00  1.0      1 -0.555570  0.831470      0.0      1.0   
2019-01-08 06:00:00+00:00  1.0      1 -0.500000  0.866025      0.0      1.0   
2019-01-08 06:15:00+00:00  1.0      1 -0.442289  0.896873      0.0      1.0   

                           month_sin  month_cos  is_weekend  is_holiday  ...  \
2019-01-08 05:15:00+00:00        0.5   0.866025           0           0  ...   
2019-01-08 05:30:00+00:00        0.5   0.866025           0           0  ...   
2019-01-08 05:45:00+00:00        0.5   0.866025           0           0  ...   
2019-01-08 06:00:00+00:00        0.5   0.866025           0           0  ...   
2019-01-08 06:15:00+00:00        0.5   0.866025           0           0  ...   

                           lag_96  lag_672  